In [1]:
import os
import json
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_from_disk
from transformers import (
    AutoModel,
    get_linear_schedule_with_warmup
)

from tqdm.auto import tqdm
from pathlib import Path

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("PyTorch version:", torch.__version__)

Device: mps
PyTorch version: 2.15.0


In [2]:
BERT_MODEL = "bert-base-uncased"
VIT_MODEL = "google/vit-base-patch16-224"

BATCH_SIZE = 8
EPOCHS = 10
PATIENCE = 4

BERT_LR = 2e-5
VIT_LR = 5e-6
FUSION_LR = 5e-4

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

DATA_PATH = Path("../data/processed/scienceqa_preprocessed")
RESULTS_PATH = Path("../results/intermediate_fusion_v1")

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print("Configuration ready.")

Configuration ready.


In [3]:
dataset = load_from_disk(str(DATA_PATH))

print(dataset)

print("Training records:", len(dataset["train"]))
print("Validation records:", len(dataset["validation"]))
print("Testing records:", len(dataset["test"]))

DatasetDict({
    train: Dataset({
        features: ['pixel_values', 'input_ids', 'attention_mask', 'option_mask', 'labels'],
        num_rows: 3530
    })
    validation: Dataset({
        features: ['pixel_values', 'input_ids', 'attention_mask', 'option_mask', 'labels'],
        num_rows: 1343
    })
    test: Dataset({
        features: ['pixel_values', 'input_ids', 'attention_mask', 'option_mask', 'labels'],
        num_rows: 2017
    })
})
Training records: 3530
Validation records: 1343
Testing records: 2017


In [4]:
REQUIRED_COLUMNS = [
    "input_ids",
    "attention_mask",
    "pixel_values",
    "option_mask",
    "labels"
]

for split in ["train", "validation", "test"]:
    missing = [
        col for col in REQUIRED_COLUMNS
        if col not in dataset[split].column_names
    ]

    if missing:
        raise ValueError(
            f"{split} is missing columns: {missing}"
        )

    dataset[split].set_format(
        type="torch",
        columns=REQUIRED_COLUMNS
    )

train_loader = DataLoader(
    dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    dataset["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    dataset["test"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 442
Validation batches: 168
Testing batches: 253


In [5]:
sample_batch = next(iter(train_loader))

for key, value in sample_batch.items():
    print(key, value.shape, value.dtype)

pixel_values torch.Size([8, 3, 224, 224]) torch.float32
input_ids torch.Size([8, 5, 128]) torch.int64
attention_mask torch.Size([8, 5, 128]) torch.int64
option_mask torch.Size([8, 5]) torch.int64
labels torch.Size([8]) torch.int64


## Initialise BERT and ViT encoders

In [6]:
bert_if1 = AutoModel.from_pretrained(
    BERT_MODEL,
    attn_implementation="eager"
)

vit_if1 = AutoModel.from_pretrained(
    VIT_MODEL,
    attn_implementation="eager"
)

# Freeze BERT
for param in bert_if1.parameters():
    param.requires_grad = False

# Fine-tune final four BERT layers
for layer in bert_if1.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Freeze ViT
for param in vit_if1.parameters():
    param.requires_grad = False

# Fine-tune final two ViT layers
for layer in vit_if1.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

bert_if1 = bert_if1.to(device)
vit_if1 = vit_if1.to(device)

print(
    "BERT trainable parameters:",
    sum(p.numel() for p in bert_if1.parameters()
        if p.requires_grad)
)

print(
    "ViT trainable parameters:",
    sum(p.numel() for p in vit_if1.parameters()
        if p.requires_grad)
)

W1111 17:59:08.135000 12039 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT trainable parameters: 28351488
ViT trainable parameters: 14175744


In [7]:
class IntermediateFusionModelV1(nn.Module):

    def __init__(self, bert, vit):
        super().__init__()

        self.bert = bert
        self.vit = vit

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=768,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )

        self.fusion_norm = nn.LayerNorm(1536)

        self.fusion = nn.Sequential(
            nn.Linear(1536, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, 1)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        option_mask
    ):

        batch_size, num_choices, seq_length = input_ids.shape

        # Flatten answer options for BERT
        flat_input_ids = input_ids.reshape(
            batch_size * num_choices,
            seq_length
        )

        flat_attention_mask = attention_mask.reshape(
            batch_size * num_choices,
            seq_length
        )

        # Extract text token representations
        text_outputs = self.bert(
            input_ids=flat_input_ids,
            attention_mask=flat_attention_mask
        )

        text_tokens = text_outputs.last_hidden_state

        # Extract image patch representations
        image_outputs = self.vit(
            pixel_values=pixel_values
        )

        image_tokens = image_outputs.last_hidden_state

        # Repeat image tokens for each answer option
        image_tokens = image_tokens.repeat_interleave(
            num_choices,
            dim=0
        )

        # Cross-attention: text queries image patches
        attended_image, _ = self.cross_attention(
            query=text_tokens,
            key=image_tokens,
            value=image_tokens,
            need_weights=False
        )

        # Mean pooling over valid text tokens
        text_mask = flat_attention_mask.unsqueeze(-1).float()

        text_features = (
            (text_tokens * text_mask).sum(dim=1)
            / text_mask.sum(dim=1).clamp(min=1)
        )

        # Mean pooling over attended visual representations
        image_features = attended_image.mean(dim=1)

        # Combine text and visual features
        combined_features = torch.cat(
            [text_features, image_features],
            dim=1
        )

        combined_features = self.fusion_norm(
            combined_features
        )

        # Score answer options
        scores = self.fusion(combined_features)

        scores = scores.reshape(
            batch_size,
            num_choices
        )

        # Exclude padded answer options
        scores = scores.masked_fill(
            option_mask == 0,
            float("-inf")
        )

        return scores

In [8]:
intermediate_fusion_v1 = IntermediateFusionModelV1(
    bert=bert_if1,
    vit=vit_if1
).to(device)

total_params = sum(
    p.numel()
    for p in intermediate_fusion_v1.parameters()
)

trainable_params = sum(
    p.numel()
    for p in intermediate_fusion_v1.parameters()
    if p.requires_grad
)

print("Model initialised.")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Model initialised.
Total parameters: 199,156,993
Trainable parameters: 45,812,737


Testing a Forward pass

In [9]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
pixel_values = batch["pixel_values"].to(device)
option_mask = batch["option_mask"].to(device)

intermediate_fusion_v1.eval()

with torch.no_grad():
    sample_scores = intermediate_fusion_v1(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        option_mask=option_mask
    )

print("Input shape:", input_ids.shape)
print("Image shape:", pixel_values.shape)
print("Output shape:", sample_scores.shape)
print("Predictions:", sample_scores.argmax(dim=1))

Input shape: torch.Size([8, 5, 128])
Image shape: torch.Size([8, 3, 224, 224])
Output shape: torch.Size([8, 5])
Predictions: tensor([3, 1, 0, 1, 3, 2, 1, 0], device='mps:0')


I use cross-entropy loss for multiple-choice classification and AdamW for optimisation. I assign separate learning rates to BERT, ViT and the cross-attention classification layers because their training requirements differ.

In [10]:
criterion_if1 = nn.CrossEntropyLoss()

optimizer_if1 = torch.optim.AdamW(
    [
        {
            "params": [
                p for p in intermediate_fusion_v1.bert.parameters()
                if p.requires_grad
            ],
            "lr": BERT_LR
        },
        {
            "params": [
                p for p in intermediate_fusion_v1.vit.parameters()
                if p.requires_grad
            ],
            "lr": VIT_LR
        },
        {
            "params": list(
                intermediate_fusion_v1.cross_attention.parameters()
            ) + list(
                intermediate_fusion_v1.fusion_norm.parameters()
            ) + list(
                intermediate_fusion_v1.fusion.parameters()
            ),
            "lr": FUSION_LR
        }
    ],
    weight_decay=WEIGHT_DECAY
)

total_training_steps = len(train_loader) * EPOCHS

warmup_steps = int(
    total_training_steps * WARMUP_RATIO
)

scheduler_if1 = get_linear_schedule_with_warmup(
    optimizer_if1,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Optimizer and scheduler configured.")
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Optimizer and scheduler configured.
Total training steps: 4420
Warmup steps: 442


In [11]:
def train_one_epoch_if1(
    model,
    loader,
    optimizer,
    scheduler,
    criterion
):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        option_mask = batch["option_mask"].to(device)
        labels = batch["labels"].to(device).long()

        optimizer.zero_grad(set_to_none=True)

        scores = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            option_mask=option_mask
        )

        loss = criterion(scores, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        optimizer.step()
        scheduler.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size

        predictions = scores.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_loss = total_loss / total
    accuracy = correct / total

    return average_loss, accuracy

In [12]:
def evaluate_if1(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc="Evaluating",
            leave=False
        ):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            option_mask = batch["option_mask"].to(device)
            labels = batch["labels"].to(device).long()

            scores = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                option_mask=option_mask
            )

            loss = criterion(scores, labels)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size

            predictions = scores.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += batch_size

    average_loss = total_loss / total
    accuracy = correct / total

    return average_loss, accuracy
    

In [13]:
history_if1 = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss_if1 = float("inf")
best_epoch_if1 = 0
patience_counter_if1 = 0

best_model_path_if1 = (
    RESULTS_PATH / "best_intermediate_fusion_v1.pth"
)

best_state_dict_if1 = None

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_accuracy = train_one_epoch_if1(
        intermediate_fusion_v1,
        train_loader,
        optimizer_if1,
        scheduler_if1,
        criterion_if1
    )

    val_loss, val_accuracy = evaluate_if1(
        intermediate_fusion_v1,
        val_loader,
        criterion_if1
    )

    history_if1["train_loss"].append(train_loss)
    history_if1["train_accuracy"].append(train_accuracy)
    history_if1["val_loss"].append(val_loss)
    history_if1["val_accuracy"].append(val_accuracy)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    if val_loss < best_val_loss_if1:

        best_val_loss_if1 = val_loss
        best_epoch_if1 = epoch + 1
        patience_counter_if1 = 0

        best_state_dict_if1 = {
            key: value.detach().cpu().clone()
            for key, value in intermediate_fusion_v1.state_dict().items()
        }

        torch.save(
            best_state_dict_if1,
            best_model_path_if1
        )

        print("Best checkpoint updated.")

    else:

        patience_counter_if1 += 1

        print(
            f"No improvement: "
            f"{patience_counter_if1}/{PATIENCE}"
        )

    if patience_counter_if1 >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining completed.")
print("Best epoch:", best_epoch_if1)
print(f"Best validation loss: {best_val_loss_if1:.4f}")


Epoch 1/10


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/168 [00:00<?, ?it/s]

Train Loss: 1.1840
Train Accuracy: 0.3425
Validation Loss: 0.9821
Validation Accuracy: 0.5577
Best checkpoint updated.

Epoch 2/10


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/168 [00:00<?, ?it/s]

... (21 outputs left out)

Train Loss: 0.2439
Train Accuracy: 0.8935
Validation Loss: 0.5567
Validation Accuracy: 0.8057
No improvement: 3/4

Epoch 10/10


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/168 [00:00<?, ?it/s]

Train Loss: 0.2377
Train Accuracy: 0.8997
Validation Loss: 0.5788
Validation Accuracy: 0.8049
No improvement: 4/4
Early stopping triggered.

Training completed.
Best epoch: 6
Best validation loss: 0.5209


In [15]:

# Evaluate the best Intermediate Fusion checkpoint

import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Recreate the model using the original notebook variables
intermediate_fusion_v1 = IntermediateFusionModelV1(
    bert=bert_if1,
    vit=vit_if1
).to(device)

# Load the best checkpoint (selected at epoch 6)
checkpoint_path = "../results/intermediate_fusion_v1/best_intermediate_fusion_v1.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

intermediate_fusion_v1.load_state_dict(checkpoint)
intermediate_fusion_v1.eval()

criterion_if1 = nn.CrossEntropyLoss()

total_loss = 0.0
correct = 0
total = 0

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Testing Intermediate Fusion"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        option_mask = batch["option_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = intermediate_fusion_v1(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            option_mask=option_mask
        )

        loss = criterion_if1(outputs, labels)

        predictions = outputs.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_loss = total_loss / total
test_accuracy = correct / total

print("\nINTERMEDIATE FUSION — TEST RESULTS")
print("=" * 45)
print(f"Test examples : {total}")
print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print("=" * 45)


Testing Intermediate Fusion:   0%|          | 0/253 [00:00<?, ?it/s]


INTERMEDIATE FUSION — TEST RESULTS
Test examples : 2017
Test loss     : 0.6906
Test accuracy: 0.6867
Test accuracy: 68.67%


In [16]:
from pathlib import Path

results_path = Path("../results")

for folder in sorted(results_path.iterdir()):
    if folder.is_dir():
        print(f"\n{folder.name}/")

        for file in sorted(folder.iterdir()):
            print("  ", file.name)


early_fusion_v2/
   best_model.pth
   training_history.json

intermediate_fusion_v1/
   best_intermediate_fusion_v1.pth

text_only_bert/
   best_model.pth
   training_history.json


In [17]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

def calculate_metrics(y_true, y_pred, model_name):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Macro)": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall (Macro)": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1 (Macro)": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        )
    }


# Intermediate Fusion metrics
intermediate_metrics = calculate_metrics(
    all_labels,
    all_predictions,
    "Intermediate Fusion"
)

print("INTERMEDIATE FUSION — EVALUATION METRICS")
print("=" * 55)

for metric, value in intermediate_metrics.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f} ({value * 100:.2f}%)")

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))


INTERMEDIATE FUSION — EVALUATION METRICS
Accuracy: 0.6867 (68.67%)
Precision (Macro): 0.5808 (58.08%)
Recall (Macro): 0.5730 (57.30%)
F1 (Macro): 0.5754 (57.54%)
Balanced Accuracy: 0.5730 (57.30%)

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.74      0.68       713
           1       0.68      0.60      0.64       726
           2       0.74      0.70      0.72       375
           3       0.85      0.83      0.84       195
           4       0.00      0.00      0.00         8

    accuracy                           0.69      2017
   macro avg       0.58      0.57      0.58      2017
weighted avg       0.69      0.69      0.68      2017


Confusion Matrix:
[[526 148  31   8   0]
 [234 434  51   7   0]
 [ 61  38 264  12   0]
 [  8  15  10 161   1]
 [  4   2   1   1   0]]
